In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw/2018")

pressure = pd.read_csv(DATA_DIR / "2018_SCADA_Pressures.csv")
flow = pd.read_csv(DATA_DIR / "2018_SCADA_Flows.csv")
demand = pd.read_csv(DATA_DIR / "2018_SCADA_Demands.csv")
leakage = pd.read_csv(DATA_DIR / "2018_Leakages.csv")
level = pd.read_csv(DATA_DIR / "2018_SCADA_Levels.csv")

datasets = {
    "pressure": pressure,
    "flow": flow,
    "demand": demand,
    "leakage": leakage,
    "level" : level
}

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist()[:10])
    print(df.head(3))
    print("Missing values:", df.isna().sum().sum())


PRESSURE
Shape: (105120, 34)
Columns: ['Timestamp', 'n1', 'n4', 'n31', 'n54', 'n105', 'n114', 'n163', 'n188', 'n215']
             Timestamp     n1     n4    n31    n54   n105   n114   n163  \
0  2018-01-01 00:00:00  28.92  33.87  37.13  37.13  50.52  53.99  52.56   
1  2018-01-01 00:05:00  28.94  33.89  37.16  37.23  50.60  54.09  52.67   
2  2018-01-01 00:10:00  28.96  33.91  37.18  37.21  50.57  54.05  52.64   

    n188   n215  ...   n549   n613   n636   n644   n679   n722   n726   n740  \
0  55.32  39.09  ...  54.80  56.16  45.53  47.62  47.33  46.14  47.12  43.81   
1  55.42  39.10  ...  54.90  56.26  45.61  47.71  47.39  46.19  47.17  43.83   
2  55.39  39.09  ...  54.87  56.24  45.61  47.71  47.41  46.20  47.19  43.84   

    n752   n769  
0  49.17  48.50  
1  49.22  48.54  
2  49.24  48.56  

[3 rows x 34 columns]
Missing values: 0

FLOW
Shape: (105120, 4)
Columns: ['Timestamp', 'p227', 'p235', 'PUMP_1']
             Timestamp   p227   p235  PUMP_1
0  2018-01-01 00:00:00  77.

In [4]:
pressure = pressure.rename(
    columns=lambda col: f"pressure_{col}" if col != "Timestamp" else col
)

flow = flow.rename(
    columns=lambda col: f"flow_{col}" if col != "Timestamp" else col
)

demand = demand.rename(
    columns=lambda col: f"demand_{col}" if col != "Timestamp" else col
)

level = level.rename(
    columns=lambda col: f"level_{col}" if col != "Timestamp" else col
)

# Merge all measurements
combined = (
    pressure
    .merge(flow, on="Timestamp", how="inner", validate="one_to_one")
    .merge(demand, on="Timestamp", how="inner", validate="one_to_one")
    .merge(level, on="Timestamp", how="inner", validate="one_to_one")
)

print(combined.shape)
print(combined.columns.tolist())


(105120, 120)
['Timestamp', 'pressure_n1', 'pressure_n4', 'pressure_n31', 'pressure_n54', 'pressure_n105', 'pressure_n114', 'pressure_n163', 'pressure_n188', 'pressure_n215', 'pressure_n229', 'pressure_n288', 'pressure_n296', 'pressure_n332', 'pressure_n342', 'pressure_n410', 'pressure_n415', 'pressure_n429', 'pressure_n458', 'pressure_n469', 'pressure_n495', 'pressure_n506', 'pressure_n516', 'pressure_n519', 'pressure_n549', 'pressure_n613', 'pressure_n636', 'pressure_n644', 'pressure_n679', 'pressure_n722', 'pressure_n726', 'pressure_n740', 'pressure_n752', 'pressure_n769', 'flow_p227', 'flow_p235', 'flow_PUMP_1', 'demand_n1', 'demand_n2', 'demand_n3', 'demand_n4', 'demand_n6', 'demand_n7', 'demand_n8', 'demand_n9', 'demand_n10', 'demand_n11', 'demand_n13', 'demand_n16', 'demand_n17', 'demand_n18', 'demand_n19', 'demand_n20', 'demand_n21', 'demand_n22', 'demand_n23', 'demand_n24', 'demand_n25', 'demand_n26', 'demand_n27', 'demand_n28', 'demand_n29', 'demand_n30', 'demand_n31', 'deman

In [5]:
duplicate_rows = combined[combined.duplicated()]
print(duplicate_rows)

Empty DataFrame
Columns: [Timestamp, pressure_n1, pressure_n4, pressure_n31, pressure_n54, pressure_n105, pressure_n114, pressure_n163, pressure_n188, pressure_n215, pressure_n229, pressure_n288, pressure_n296, pressure_n332, pressure_n342, pressure_n410, pressure_n415, pressure_n429, pressure_n458, pressure_n469, pressure_n495, pressure_n506, pressure_n516, pressure_n519, pressure_n549, pressure_n613, pressure_n636, pressure_n644, pressure_n679, pressure_n722, pressure_n726, pressure_n740, pressure_n752, pressure_n769, flow_p227, flow_p235, flow_PUMP_1, demand_n1, demand_n2, demand_n3, demand_n4, demand_n6, demand_n7, demand_n8, demand_n9, demand_n10, demand_n11, demand_n13, demand_n16, demand_n17, demand_n18, demand_n19, demand_n20, demand_n21, demand_n22, demand_n23, demand_n24, demand_n25, demand_n26, demand_n27, demand_n28, demand_n29, demand_n30, demand_n31, demand_n32, demand_n33, demand_n34, demand_n35, demand_n36, demand_n39, demand_n40, demand_n41, demand_n42, demand_n43, dem

In [6]:
pipe_columns = leakage.columns.drop("Timestamp")

leakage["Total_Leakage"] = leakage[pipe_columns].sum(
    axis=1,
    min_count=1
)
leakage_target = leakage[["Timestamp", "Total_Leakage"]]

leakage_target.head()

,Timestamp,Total_Leakage
0,2018-01-01 00:00:00,0.0
1,2018-01-01 00:05:00,0.0
2,2018-01-01 00:10:00,0.0
3,2018-01-01 00:15:00,0.0
4,2018-01-01 00:20:00,0.0


In [7]:
df = (
    combined
    .merge(leakage_target, on="Timestamp", how="inner", validate="one_to_one")
)
 
print(df.shape)
df.head()

(105120, 121)


,Timestamp,pressure_n1,pressure_n4,pressure_n31,pressure_n54,pressure_n105,pressure_n114,pressure_n163,pressure_n188,pressure_n215,...,demand_n382,demand_n383,demand_n384,demand_n385,demand_n386,demand_n387,demand_n388,demand_n389,level_T1,Total_Leakage
0,2018-01-01 00:00:00,28.92,33.87,37.13,37.13,50.52,53.99,52.56,55.32,39.09,...,67.61,65.82,65.25,57.39,214.20,81.49,256.50,92.14,3.50,0.0
1,2018-01-01 00:05:00,28.94,33.89,37.16,37.23,50.60,54.09,52.67,55.42,39.10,...,95.29,50.18,81.29,49.67,109.14,104.46,394.48,203.61,3.51,0.0
2,2018-01-01 00:10:00,28.96,33.91,37.18,37.21,50.57,54.05,52.64,55.39,39.09,...,178.10,63.74,252.94,9.69,285.82,94.32,337.33,93.11,3.53,0.0
3,2018-01-01 00:15:00,28.98,33.93,37.19,37.27,50.60,54.08,52.70,55.45,39.09,...,51.60,33.93,139.25,36.15,183.66,57.27,268.74,96.19,3.54,0.0
4,2018-01-01 00:20:00,28.98,33.93,37.19,37.14,50.53,54.00,52.57,55.32,39.09,...,79.05,68.27,43.76,22.05,226.42,140.88,423.29,96.87,3.56,0.0


In [8]:
# Missing values
print(df.isna().sum().sum())

# Duplicated timestamps
print(df["Timestamp"].duplicated().sum())

# Numerical summary
print(df.describe())

# Negative leakage values
print((df["Total_Leakage"] < 0).sum())

# Check timeline continuity
timestamps = pd.to_datetime(df["Timestamp"]).sort_values()

print(timestamps.diff().value_counts().head())

0
0
         pressure_n1    pressure_n4   pressure_n31   pressure_n54  \
count  105120.000000  105120.000000  105120.000000  105120.000000   
mean       28.177514      33.264552      36.604953      36.796791   
std         0.548432       0.541586       0.508478       0.739204   
min        25.670000      30.900000      34.590000      34.330000   
25%        27.740000      32.840000      36.190000      36.190000   
50%        28.100000      33.200000      36.550000      36.810000   
75%        28.630000      33.720000      37.030000      37.370000   
max        29.360000      34.310000      37.570000      38.290000   

       pressure_n105  pressure_n114  pressure_n163  pressure_n188  \
count  105120.000000  105120.000000  105120.000000  105120.000000   
mean       50.243409      53.659096      52.139037      54.889171   
std         0.358803       0.420987       0.606962       0.600544   
min        48.770000      51.970000      49.770000      52.480000   
25%        49.990000      53.